# Workflow 07

## Configuration

In [1]:
chemfilesdir = "chemfiles/"
resultsdir = "results/"
docked = resultsdir + "docked.sdf"
log = resultsdir + "gninalog.txt"
rmsdlog = resultsdir + "rmsd.txt"
masterlog = resultsdir + "masterlog.txt"

In [2]:
experimentdescription = "Workflow 7 is a refactor of Workflow 6. It includes the addition of all protein and ligand details in dictionary objects."

In [3]:
proteindict = {}
liganddict = {}
gninaoptdict = {}
prepdict = {}

In [4]:
import copy
def reportdict(rows, columns):
    lines = []
    if len(rows) == 0:
        return (lines) 
    inner_keys = set()
    for r in rows.values():
        inner_keys.update(r.keys())
    col_widths = {}
    col_widths["id"] = max(len("id"), max(len(k) for k in rows))
    for col in inner_keys:
        header_len = len(col)
        data_len = max(len(str(r.get(col, ""))) for r in rows.values())
        col_widths[col] = max(header_len, data_len)
    header = "  ".join(f"{col:<{col_widths[col]}}" for col in columns)
    lines.append(header)
    for outer_key, inner in rows.items():
        cells = [f"{outer_key:<{col_widths['id']}}"]
        for col in columns[1:]:
            cells.append(f"{str(inner.get(col, '')):<{col_widths[col]}}")
        lines.append("  ".join(cells))
    return (lines)

## Create test ligand from SMILES

In [5]:
ligand = {}
ligand["code"] = "E20"
ligand["name"] = "E20 created from SMILES"
ligand["smile"] = "COc1cc2c(cc1OC)C(=O)[C@@H](C2)CC3CCN(CC3)Cc4ccccc4"
ligand["source"] = "https://www.ebi.ac.uk/pdbe-srv/pdbechem/chemicalCompound/show/E20#structures-section"
ligand["filename"] = "E20.sdf"
ligand["prep"] = "obabel --gen3d"
liganddict["smilesE20"] = ligand

In [6]:
# Prepare one or many ligands with Openbabel.
# Loop over keys and values
for key, value in liganddict.items():
    smilessdf = chemfilesdir + value["filename"]
    !obabel -:"{value["smile"]}" --gen3d -osdf -O"{smilessdf}"

1 molecule converted


#### Add SMILES preparation step to list of preparations ####

In [7]:
prep = {}
prep["name"] = "obabel --gen3d"
prep["description"] = "OpenBabel executable used to convert SMILES chain into 3D .sdf file via '--gen3d' flag."
prep["functionname"] = "TBD"
prepdict["obabel --gen3d"] = prep

## Populate protein dictionary and then download files from protein database websites ##

In [8]:
protein = {}
protein["code"] = "6o4w"
protein["id"] = "6o4wREDO"
protein["name"] = "6o4w from RDBSREDO"
protein["keepchain"] = "B"
protein["keephet"] = "E20"
protein["source"] = "https://pdb-redo.eu/db/6o4w/6o4w_final.pdb"
protein["filenameroot"] = "6o4w_redo"
proteindict ["REDO6o4w"] = protein

In [9]:
protein = {}
protein["code"] = "6O4W"
protein["id"] = "6O4WRCBS"
protein["name"] = "6O4W from RCBS"
protein["keepchain"] = "B"
protein["keephet"] = "E20"
protein["source"] = "https://files.rcsb.org/download/6O4W.pdb"
protein["filenameroot"] = "6O4W_rcbs"
proteindict ["RCBS6O4W"] = protein

In [10]:
# Loop over keys and values
for key, value in proteindict.items():
    localfilename = chemfilesdir + value['filenameroot'] + ".pdb"
    !wget -nv -O "{localfilename}" "{value['source']}"
    

2026-01-29 11:45:23 URL:https://pdb-redo.eu/db/6o4w/6o4w_final.pdb [733809] -> "chemfiles/6o4w_redo.pdb" [1]
2026-01-29 11:45:23 URL:https://files.rcsb.org/download/6O4W.pdb [775980] -> "chemfiles/6O4W_rcbs.pdb" [1]


## Examine proteins to determine what chains and ligands are present in the proteins. (Optional informational step) ##

In [ ]:
import gemmi

for key, value in proteindict.items():
    
    structure = gemmi.read_structure(chemfilesdir + value['filenameroot'] + ".pdb")
    #structure = gemmi.read_structure(chemfilesdir + "6O4w_rcbs.pdb")
    ligands = []
    
    for model in structure:
        for chain in model:
            for res in chain:
                if res.het_flag != ' ':  # hetero-residue
                    if res.name not in ("HOH", "WAT", "H2O"):
                        if res.seqid.num == 604:
                            ligands.append((res.name, chain.name, res.seqid.num))

    print("protein: " + value['filenameroot'])
    print(set(ligands))

## Remove unneccessary chains ##

In [11]:
#Chainstripper
from Bio.PDB import PDBParser, PDBIO, Select

def ChainStripper(protein):
    print ("ChainStripper: " + protein["name"])
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("struct", chemfilesdir + protein["filenameroot"] + ".pdb")
    io = PDBIO()
    io.set_structure(structure)
    io.save(chemfilesdir + protein["filenameroot"] + "_" + protein["keepchain"] + ".pdb", KeepChain(protein["keepchain"])) 

def KeepChain(chain_id):
    class KeepNon(Select):
        def accept_chain(self, chain):
            return chain.id == chain_id
    return KeepNon()

for key, value in proteindict.items():
    ChainStripper(value)
    

ChainStripper: 6o4w from RDBSREDO
ChainStripper: 6O4W from RCBS


## Extract the bound ligand

In [ ]:
# Notes about obabel.
# the -L flag does not seem to care what 'label' it is passed.
# I beleive the .cif has at least three ligands so I am not confident this code creates a ligand file with only E20.
# It appears the '-l' flag is wholly ignored. The output file type seems to instruct obabel how it should modify the molecule.
#!obabel "workflow03/6o4w_final.cif" -l "LIG" -O "workflow03/6o4w_final_ligand.sdf"
#!obabel "workflow03/6o4w_final.cif" -l "E20" -osdf -O "workflow03/6o4w_final_ligand_E20.sdf"
#!obabel "workflow03/6o4w_final_chainB.pdb" -l "E20" -osdf -O "workflow03/6o4w_final_chainB_ligand_E20.sdf"

#I'm not convinced the following code from Bio is the best ligand extraction. More research necessary in the future.

In [12]:
# Extract ligand based on ligand code.

from Bio.PDB import PDBParser, PDBIO, Select

class LigandSelect(Select):
    def __init__(self, ligand_name, waters=None):
        self.ligand_name = ligand_name.upper()
        self.waters = waters or {"HOH", "WAT", "H2O"}

    def accept_residue(self, residue):
        hetflag, resseq, icode = residue.id
        resname = residue.get_resname().strip().upper()
        # Hetero residue with the desired name, not water
        if hetflag != " " and resname == self.ligand_name and resname not in self.waters:
            return True
        return False

    def accept_atom(self, atom):
        # Keep all atoms of accepted residues
        return True

def extract_ligand_by_resname(in_pdb, out_pdb, ligand_code):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("struct", in_pdb)
    io = PDBIO()
    io.set_structure(structure)
    io.save(out_pdb, LigandSelect(ligand_code))

In [13]:
from rdkit import Chem

for key, value in proteindict.items():
    curfilename = value["filenameroot"] + "_" + value["keepchain"] + ".pdb"
    newfilename = value["filenameroot"] + "_" + value["keepchain"] + "_" + value["keephet"] + ".sdf"
    
    extract_ligand_by_resname(
        in_pdb= chemfilesdir + curfilename,
        out_pdb= chemfilesdir + newfilename,
        ligand_code = value["keephet"],   
    ) 

    mol = Chem.MolFromPDBFile(chemfilesdir + newfilename, removeHs=False) #removeHS=False preserves all hydrogens
    w = Chem.SDWriter(chemfilesdir + newfilename)
    w.write(mol)
    w.close()

    ligand = {}
    ligand["code"] = value["keephet"]
    ligand["name"] = value["keephet"] + " extracted from protein " + newfilename
    ligand["filename"] = newfilename
    ligand["source"] = curfilename
    ligand["prep"] = "rdkit_save_H"
    ligandcode = value["id"] + "complexed" + value["keephet"]
    liganddict[ligandcode] = ligand
    
    print(curfilename + " -> " + newfilename)

6o4w_redo_B.pdb -> 6o4w_redo_B_E20.sdf
6O4W_rcbs_B.pdb -> 6O4W_rcbs_B_E20.sdf


In [14]:
lines = reportdict(liganddict, ["code","name","source"])
print("\n".join(lines))

code  name                                            source                                                                              
smilesE20             E20 created from SMILES                         https://www.ebi.ac.uk/pdbe-srv/pdbechem/chemicalCompound/show/E20#structures-section
6o4wREDOcomplexedE20  E20 extracted from protein 6o4w_redo_B_E20.sdf  6o4w_redo_B.pdb                                                                     
6O4WRCBScomplexedE20  E20 extracted from protein 6O4W_rcbs_B_E20.sdf  6O4W_rcbs_B.pdb                                                                     


#### Add SMILES preparation step to list of preparations

In [15]:
prep = {}
prep["name"] = "rdkit_save_H"
prep["description"] = "Python 'Bio.PDB' package used to remove unnecessary chain(s) from the protein file."
prep["description"] += " Python 'Bio.PDB' package used to extract the ligand via code name."
prep["description"] += " Python 'rdkit' package used to convert the ligand file from PDB file to SDF file format."
prep["functionname"] = "TBD"
prepdict["rdkit_save_H"] = prep

## Fix the protein

In [16]:
#PDBFixer python package is required by the following block. 
#This code expects that PDBFixer package is installed in the Conda environment that Jupyter is running within.
from pdbfixer import PDBFixer
from openmm.app import PDBFile

In [17]:
# define the PDBFixer function
# Options must be called in this specific order. Some options rely on earlier options.
# mode: 0 = rewrite no tasks, 1 = removeHeterogens only, 2 = Also hydrogens, 3 = full set of Andy tasks, 4 full set minus Hydrogen

def PDBfixfunc(localprotein, savelocation, mode = 0):

    fixer = PDBFixer(filename = localprotein)

    if (mode >= 2 ):
      #fixer.removeChains(indices) - never
      fixer.findMissingResidues()

    if (mode >= 3 ):
      fixer.findNonstandardResidues()
      fixer.replaceNonstandardResidues()

    if (mode == 1 or mode >= 3 ):
      fixer.removeHeterogens(False) #removes the complexed ligand
      # The argument specifies whether to keep water molecules. 
      # False removes all heterogens including water. True keeps water molecules while removing all other heterogens. 

    if (mode >= 2 and mode <= 3 ):
      fixer.findMissingAtoms() # required by addMissingAtoms
      fixer.addMissingAtoms() # required by addMissingHydrogens
      fixer.addMissingHydrogens(7.0)
      #fixer.addSolvent() - never
      #fixer.addMembrane('POPE') - never

    #PDBFile.writeFile(fixer.topology, fixer.positions, open(savelocation, 'w'))
    PDBFile.writeFile(fixer.topology, fixer.positions, open(savelocation, 'w'), keepIds=True)

    del(fixer)

def PDBfixfunc74(localprotein, savelocation):
    fixer = PDBFixer(filename = localprotein)
    fixer.findMissingResidues()
    fixer.findNonstandardResidues()
    fixer.replaceNonstandardResidues()
    fixer.removeHeterogens(False) #removes the complexed ligand
    fixer.findMissingAtoms() # required by addMissingAtoms
    fixer.addMissingAtoms() # required by addMissingHydrogens
    fixer.addMissingHydrogens(7.4)
    PDBFile.writeFile(fixer.topology, fixer.positions, open(savelocation, 'w'), keepIds=True)


In [18]:
prep = {}
prep["name"] = "PDBfixfunc74"
prep["description"] = "Python 'Bio.PDB' package Chainstripper() function used to remove extra chains."
prep["description"] += " PDBFixer package used to fix the protein with following options:"
prep["description"] += " findMissingResidues(), findNonstandardResidues(), replaceNonstandardResidues(), fixer.removeHeterogens(False),"
prep["description"] += " findMissingAtoms(), addMissingAtoms(), addMissingHydrogens(7.4)"
prep["functionname"] = "PDBfixfunc74()"
prepdict["PDBfixfunc"] = prep

In [31]:
for key, value in proteindict.items():
    curfilename = value["filenameroot"] + "_" + value["keepchain"] + ".pdb"
    newfilename = value["filenameroot"] + "_" + value["keepchain"] + "_fixed.pdb"
    PDBfixfunc74(chemfilesdir + curfilename,chemfilesdir + newfilename)
    value["filename"] = newfilename
    value["prep"] = "PDBfixfunc74"
    print("PDBfixfunc74() " + curfilename + " => " + newfilename)

PDBfixfunc74() 6o4w_redo_B.pdb => 6o4w_redo_B_fixed.pdb
PDBfixfunc74() 6O4W_rcbs_B.pdb => 6O4W_rcbs_B_fixed.pdb


In [32]:
lines = reportdict(proteindict, ["code","id","name","prep","filename"])
print("\n".join(lines))

code  id        name                prep          filename             
REDO6o4w  6o4wREDO  6o4w from RDBSREDO  PDBfixfunc74  6o4w_redo_B_fixed.pdb
RCBS6O4W  6O4WRCBS  6O4W from RCBS      PDBfixfunc74  6O4W_rcbs_B_fixed.pdb


## Setup scoring function and results structure

In [21]:
# Get values from line 1 of results section from GNINA results file.
def get_mode1_values(path):
    mode_section = False
    with open(path) as f:
        mode_section = False
        for line in f:
            line = line.strip()
            if line.startswith("mode"):
                mode_section = True
            if not line or line.startswith("mode") or line.startswith("-") or line.startswith("|") or mode_section is False:
                continue
            parts = line.split()
            mode = int(parts[0])
            if mode == 1:
                affinity = float(parts[1])
                intramol = float(parts[2])
                cnn_pose = float(parts[3])
                cnn_aff = float(parts[4])
                return mode, affinity, intramol, cnn_pose, cnn_aff
# Example usage:   mode, affinity, intramol, cnn_pose, cnn_aff = get_mode1_values("workflow03/log.txt")

In [22]:
opt = "--exhaustiveness=16 --num_modes=8 --seed 0 --pose_sort_order CNNaffinity --no_gpu" 

gninaopt = {}
gninaopt["opt"] = opt
gninaoptdict["ex16nm8seed0CNN--no_gpu"] = gninaopt

## GNINA and reporting functions

In [ ]:
#Examples
#output = !~/gnina-binary/gnina -r proteins_pdbfixtest/2ama_fixer_andy_minusH.pdb -l proteins_pdbfixtest/Trenbolone.sdf --autobox_ligand proteins_pdbfixtest/2ama-ligand.pdb -o docked.sdf.gz  --exhaustiveness=64 --num_modes=8 --seed 0 --pose_sort_order CNNaffinity --log proteins_pdbfixtest/log.txt
# Predict against complexed ligand.
#!~/octoberproject/gnina -r 8GUTciffixed4.pdb -l 8GUT-ligand.sdf --autobox_ligand 8GUT-ligand.sdf -o docked.sdf.gz  --exhaustiveness=64 --num_modes=8 --seed 0 --pose_sort_order CNNaffinity --log log.txt --no_gpu  

In [23]:
# Call Gnina, parse results from RMSD calculation, call function that parse results from both log files and appends to results variable

#pro = subdir + "6O4w_rcbs_chainB_fixed.pdb"
#lig = subdir + "6O4w_rcbs_chainB_E20.sdf"
#box = subdir + "6O4w_rcbs_chainB_E20.sdf"
#docked = subdir + "docked.sdf"
#log = subdir + "gninalog.txt"
#!~/octoberproject/gnina -r "{pro}" -l "{lig}" --autobox_ligand "{box}" -o "{docked}" --log "{log}" --exhaustiveness=16 --num_modes=8 --seed 0 --pose_sort_order CNNaffinity --no_gpu  

def rungnina(proteinid,ligandid,boxid):
    protein = proteindict[proteinid]
    ligand = liganddict[ligandid]
    box = liganddict[boxid]
    p = chemfilesdir + protein["filename"]
    l = chemfilesdir + ligand["filename"]
    b = chemfilesdir + box["filename"]
    !~/octoberproject/gnina -r "{p}" -l "{l}" --autobox_ligand "{b}" -o "{docked}" --log "{log}" --exhaustiveness=16 --num_modes=8 --seed 0 --pose_sort_order CNNaffinity --no_gpu  

    #Compute RMSD of two docked ligand files and save results to rmsdlog file.
    !obrms -f "{b}" "{docked}" | tee "{rmsdlog}"

    appendresults(proteinid,ligandid,boxid)

In [24]:
# Get values from text files, append to results data, print to screen.
# Note: The CNN_VS score is also report in the docked.sdf file.

def appendresults(pro, lig, box):

    mode, affinity, intramol, cnnpose, cnnaffinity = get_mode1_values(log)
    cnnvs = cnnpose * cnnaffinity
    
    with open(rmsdlog) as f:
        first_line = f.readline()
        line = first_line.strip()
        parts = line.split()
        rmsd = float(parts[2])
    new_row = (pro + " + " + lig + " @ " + box, cnnpose, cnnvs, rmsd)
    data.append(new_row)

In [25]:
def reporttable(rows):
    lines = []
    # rows: list of tuples, all same length
    if not rows:
        return
    # Number of columns
    n_cols = len(rows[0])
    # Max width per column (over all rows)
    widths = [
        max(len(str(row[col])) for row in rows)
        for col in range(n_cols)
    ]
    # Print each row with aligned columns
    for row in rows:
        line = "  ".join(
            f"{str(value):<{widths[i]}}"
            for i, value in enumerate(row)
        )
        #print(line)
        lines.append(line)

    return (lines)

In [26]:
#Report out results with details of Ligands and Proteins used
from datetime import datetime

now = datetime.now()
stamp = now.strftime("%Y-%m-%d %H:%M:%S")
unixtime = int(datetime.now().timestamp())

def reportall():
    lines = []

    lines.append("Report produced " + stamp + " experiment id: " + str(unixtime))

    lines.append("")
    lines.append(experimentdescription)

    lines.append("")
    cols = ["name","functionname","description"]
    lines.append("File preparation descriptions:")
    lines = lines + reportdict(prepdict,cols)
                
    lines.append("")
    cols = ["id", "name", "prep"]
    lines.append("Proteins:")
    lines = lines + reportdict(proteindict, cols)

    lines.append("")
    lines.append("Ligands:")
    lines = lines + reportdict(liganddict, cols)

    lines.append("")
    cols = ["opt"]
    lines.append("Gnina options:")
    lines = lines + reportdict(gninaoptdict,cols)

    lines.append("")
    lines.append("Results:")
    newrow = ("proteinid + ligandid @ ligandid_searchspace", "CNN Score", "CNN_VS", "RMSD") 
    outputlist = list(data)
    outputlist.insert(0, newrow)
    lines = lines + reporttable(outputlist)

    return lines;

In [ ]:
out = reportall()
print (out)


## Call Gnina

In [37]:
#populate first row with comparison data from Bucchari paper.
data = [
    ("Bucchari (PDB-REDO)", "0.92", "7.29", "1.71"),
]

In [38]:
#1
#rungnina("protein_id", "ligand_id", "ligand_id_for_searchspace")
rungnina("REDO6o4w", "6o4wREDOcomplexedE20", "6o4wREDOcomplexedE20")

              _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina v1.3.2 master:f23dd2b   Built Jul 29 2025.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: /home/dwaine/octoberproject/gnina -r chemfiles/6o4w_redo_B_fixed.pdb -l chemfiles/6o4w_redo_B_E20.sdf --autobox_ligand chemfiles/6o4w_redo_B_E20.sdf -o results/docked.sdf --log results/gninalog.txt --exhaustiveness=16 --num_modes=8 --seed 0 --pose_sort_order CNNaffinity --no_gpu
Using random seed: 0

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

mode |  affinity  |  intramol  |    CNN     |   CNN
     | (kcal/mol) | (kcal/mol) | pose score | affinity
-----+------------+------------+------------+----------
    1    

In [39]:
#2
#rungnina("protein_id", "ligand_id", "ligand_id_for_searchspace")
rungnina("REDO6o4w", "smilesE20", "6o4wREDOcomplexedE20")

              _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina v1.3.2 master:f23dd2b   Built Jul 29 2025.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: /home/dwaine/octoberproject/gnina -r chemfiles/6o4w_redo_B_fixed.pdb -l chemfiles/E20.sdf --autobox_ligand chemfiles/6o4w_redo_B_E20.sdf -o results/docked.sdf --log results/gninalog.txt --exhaustiveness=16 --num_modes=8 --seed 0 --pose_sort_order CNNaffinity --no_gpu
Using random seed: 0

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
 | pose 0 | initial pose not within box

mode |  affinity  |  intramol  |    CNN     |   CNN
     | (kcal/mol) | (kcal/mol) | pose score | affinity
-----+------------+------------+-----

In [40]:
#3
#rungnina("protein_id", "ligand_id", "ligand_id_for_searchspace")
rungnina("RCBS6O4W", "6O4WRCBScomplexedE20", "6O4WRCBScomplexedE20")

              _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina v1.3.2 master:f23dd2b   Built Jul 29 2025.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: /home/dwaine/octoberproject/gnina -r chemfiles/6O4W_rcbs_B_fixed.pdb -l chemfiles/6O4W_rcbs_B_E20.sdf --autobox_ligand chemfiles/6O4W_rcbs_B_E20.sdf -o results/docked.sdf --log results/gninalog.txt --exhaustiveness=16 --num_modes=8 --seed 0 --pose_sort_order CNNaffinity --no_gpu
Using random seed: 0

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

mode |  affinity  |  intramol  |    CNN     |   CNN
     | (kcal/mol) | (kcal/mol) | pose score | affinity
-----+------------+------------+------------+----------
    1    

## Display all results

In [41]:
#This block presents the results to stdout.
lines = reportall()
text = "\n".join(lines) + "\n"
print(text)

Report produced 2026-01-29 11:47:45 experiment id: 1769716065

Workflow 7 is a refactor of Workflow 6. It includes the addition of all protein and ligand details in dictionary objects.

File preparation descriptions:
name            functionname    description                                                                                                                                                                                                                                                                                                                   
obabel --gen3d  TBD             OpenBabel executable used to convert SMILES chain into 3D .sdf file via '--gen3d' flag.                                                                                                                                                                                                                                       
rdkit_save_H    TBD             Python 'Bio.PDB' package used to remove unnecessa

In [42]:
#This block will append results listed above to master results file.

with open(masterlog, "a") as f:
    f.write(text)

## Maya Results from 12/15/2025

In [ ]:
https://colab.research.google.com/drive/14LTM5RHdjNY8h49LJ7C5K5_l5B_IqyTa?usp=drive_link#scrollTo=8mTtZuPU5E8Z

Results:

Method	                                CNN ScoreCNN_VS	RMSD
Bucchari (PDB-REDO)	                    0.92	7.29	1.71
PDB-REDO file with re-docking Donepezil	0.96	7.14	0.39
PDB-REDO file with generated Donepezil	0.94	6.85	1.07
RCSB PDB file with re-docking Donepezil	0.93	6.85	0.58


## Notes

In [ ]:
#!jupyter nbconvert --to html --stdout workflow02.ipynb
!jupyter nbconvert --to html workflow02.ipynb